In [ ]:
%pip install flask flask-cors

In [ ]:
from pathlib import Path
from flask import Flask, jsonify, request
from flask_cors import CORS
import sqlite3

app = Flask(__name__)
CORS(app)

WEATHER_OPTIONS = ["Hot", "Mild", "Snowy"]


def get_db_path():
    # Wrap the string in Path()
    db_path = Path("/travel_planner.db")
    
    if db_path.exists():
        return db_path.resolve()
        
    raise FileNotFoundError(f"Database not found at: {db_path}")

def get_db_connection():
    conn = sqlite3.connect(get_db_path())
    conn.row_factory = sqlite3.Row
    return conn


def weather_band_sql():
    return """
        CASE
            WHEN weather_monthly.avg_temp_c <= 5 THEN 'Snowy'
            WHEN weather_monthly.avg_temp_c >= 25 THEN 'Hot'
            ELSE 'Mild'
        END
    """


def normalize(value):
    return value.strip().lower() if isinstance(value, str) else ""

# --- UPDATE NEEDED: Change this route from '/api/destinations' to '/api/cities' ---
# Your frontend is pinging "http://127.0.0.1:5001/api/cities" (which is why you got 404 errors in the console).
# Change it to: @app.route('/api/cities', methods=['GET'])
@app.route('/api/destinations', methods=['GET'])
def get_destinations():
    # --- UPDATE NEEDED: Function name ---
    # Optional, but you might want to rename this function to `get_cities()` to match the route.
    conn = get_db_connection()
    data = conn.execute("""
        SELECT
            destinations.id,
            destinations.name,
            destinations.destination_type,
            countries.name AS country,
            countries.region,
            countries.currency_code,
            countries.main_language
        FROM destinations
        JOIN countries ON countries.id = destinations.country_id
        ORDER BY destinations.name;
    """).fetchall()
    conn.close()
    return jsonify([dict(row) for row in data])


@app.route('/api/activities', methods=['GET'])
def get_activities():
    conn = get_db_connection()
    data = conn.execute("""
        SELECT
            activities.id,
            destinations.name AS destination,
            activity_types.name AS activity_type,
            activities.activity_name
        FROM activities
        JOIN destinations ON destinations.id = activities.destination_id
        JOIN activity_types ON activity_types.id = activities.activity_type_id
        ORDER BY activity_types.name, destinations.name;
    """).fetchall()
    conn.close()
    return jsonify([dict(row) for row in data])


@app.route('/api/vibes', methods=['GET'])
def get_vibes():
    conn = get_db_connection()
    data = conn.execute("SELECT * FROM travel_vibes ORDER BY name;").fetchall()
    conn.close()
    return jsonify([dict(row) for row in data])


@app.route('/api/filter-options', methods=['GET'])
def get_filter_options():
    conn = get_db_connection()
    cost_options = [row['budget_level'] for row in conn.execute("""
        SELECT DISTINCT budget_level
        FROM cost_profiles
        WHERE budget_level IS NOT NULL
        ORDER BY budget_level;
    """).fetchall()]
    activity_options = [row['name'] for row in conn.execute("""
        SELECT DISTINCT activity_types.name
        FROM activity_types
        JOIN activities ON activities.activity_type_id = activity_types.id
        ORDER BY activity_types.name;
    """).fetchall()]
    vibe_options = [row['name'] for row in conn.execute("""
        SELECT DISTINCT travel_vibes.name
        FROM travel_vibes
        JOIN destination_vibes ON destination_vibes.vibe_id = travel_vibes.id
        ORDER BY travel_vibes.name;
    """).fetchall()]
    conn.close()

    return jsonify({
        "cost": cost_options,
        "weather": WEATHER_OPTIONS,
        "activity": activity_options,
        "vibe": vibe_options,
    })


@app.route('/api/search', methods=['GET'])
def search_destinations():
    selected_filters = {
        "cost": request.args.get("cost", "").strip(),
        "weather": request.args.get("weather", "").strip(),
        "activity": request.args.get("activity", "").strip(),
        "vibe": request.args.get("vibe", "").strip(),
    }
    selected_filters = {key: value for key, value in selected_filters.items() if value}

    conn = get_db_connection()
    rows = conn.execute(f"""
        SELECT
            destinations.id AS destination_id,
            destinations.name AS destination,
            countries.name AS country,
            cost_profiles.budget_level AS cost,
            weather_monthly.avg_temp_c,
            weather_monthly.rainfall_mm,
            {weather_band_sql()} AS weather_band,
            activity_types.name AS activity_type,
            activities.activity_name,
            travel_vibes.name AS vibe
        FROM destinations
        JOIN countries ON countries.id = destinations.country_id
        JOIN cost_profiles ON cost_profiles.destination_id = destinations.id
        JOIN weather_monthly ON weather_monthly.destination_id = destinations.id
        JOIN activities ON activities.destination_id = destinations.id
        JOIN activity_types ON activity_types.id = activities.activity_type_id
        LEFT JOIN destination_vibes ON destination_vibes.destination_id = destinations.id
        LEFT JOIN travel_vibes ON travel_vibes.id = destination_vibes.vibe_id
        ORDER BY destinations.name, travel_vibes.name;
    """).fetchall()
    conn.close()

    destinations = {}
    for row in rows:
        destination_id = row['destination_id']
        if destination_id not in destinations:
            destinations[destination_id] = {
                "destination": row['destination'],
                "country": row['country'],
                "cost": row['cost'],
                "avg_temp_c": row['avg_temp_c'],
                "rainfall_mm": row['rainfall_mm'],
                "weather_band": row['weather_band'],
                "activity_type": row['activity_type'],
                "activity_name": row['activity_name'],
                "vibes": [],
            }

        if row['vibe'] and row['vibe'] not in destinations[destination_id]["vibes"]:
            destinations[destination_id]["vibes"].append(row['vibe'])

    results = []
    selected_filter_count = len(selected_filters)

    for destination in destinations.values():
        match_count = 0

        if normalize(selected_filters.get("cost", "")) == normalize(destination["cost"]):
            match_count += 1
        if normalize(selected_filters.get("weather", "")) == normalize(destination["weather_band"]):
            match_count += 1
        if normalize(selected_filters.get("activity", "")) == normalize(destination["activity_type"]):
            match_count += 1
        if selected_filters.get("vibe") and normalize(selected_filters["vibe"]) in [normalize(vibe) for vibe in destination["vibes"]]:
            match_count += 1

        destination["match_count"] = match_count
        destination["selected_filter_count"] = selected_filter_count
        results.append(destination)

    results.sort(key=lambda destination: (-destination["match_count"], destination["destination"]))
    return jsonify(results)


if __name__ == '__main__':
    app.run(port=5001, debug=True, use_reloader=False)

In [ ]:
from pathlib import Path
from flask import Flask, jsonify, request
from flask_cors import CORS
import sqlite3

app = Flask(__name__)
CORS(app)

# The new options!
WEATHER_OPTIONS = ["Warm", "Mild", "Cold"] 

def get_db_path():
    db_path = Path("travel_planner.db")
    if db_path.exists():
        return db_path.resolve()
    raise FileNotFoundError(f"Database not found at: {db_path}")

def get_db_connection():
    conn = sqlite3.connect(get_db_path())
    conn.row_factory = sqlite3.Row
    return conn

def normalize(value):
    return value.strip().lower() if isinstance(value, str) else ""

@app.route('/api/cities', methods=['GET'])
def get_destinations():
    conn = get_db_connection()
    data = conn.execute("""
        SELECT 
            DestinationID as id, 
            CityName as name, 
            Country as country, 
            Description as description
        FROM Destinations
        ORDER BY CityName;
    """).fetchall()
    conn.close()
    return jsonify([dict(row) for row in data])

@app.route('/api/activities', methods=['GET'])
def get_activities():
    conn = get_db_connection()
    data = conn.execute("""
        SELECT Activities.ActivityID as id, Destinations.CityName AS destination,
               Activities.ActivityName AS activity_name, Destinations_Activities.Spotlight_Description as description
        FROM Activities
        JOIN Destinations_Activities ON Destinations_Activities.ActivityID = Activities.ActivityID
        JOIN Destinations ON Destinations.DestinationID = Destinations_Activities.DestinationID
        ORDER BY Activities.ActivityName, Destinations.CityName;
    """).fetchall()
    conn.close()
    return jsonify([dict(row) for row in data])

@app.route('/api/vibes', methods=['GET'])
def get_vibes():
    conn = get_db_connection()
    data = conn.execute("SELECT VibeID as id, VibeName as name FROM Travel_Vibes ORDER BY VibeName;").fetchall()
    conn.close()
    return jsonify([dict(row) for row in data])

@app.route('/api/filter-options', methods=['GET'])
def get_filter_options():
    conn = get_db_connection()
    cost_options = [row['BudgetLevel'] for row in conn.execute("SELECT DISTINCT BudgetLevel FROM Cost_Profiles WHERE BudgetLevel IS NOT NULL ORDER BY BudgetLevel;").fetchall()]
    activity_options = [row['name'] for row in conn.execute("SELECT DISTINCT ActivityName as name FROM Activities ORDER BY ActivityName;").fetchall()]
    vibe_options = [row['name'] for row in conn.execute("SELECT DISTINCT VibeName as name FROM Travel_Vibes ORDER BY VibeName;").fetchall()]
    conn.close()
    return jsonify({
        "cost": cost_options,
        "weather": WEATHER_OPTIONS,
        "activity": activity_options,
        "vibe": vibe_options,
    })

@app.route('/api/search', methods=['GET'])
def search_destinations():
    selected_filters = {
        "cost": request.args.get("cost", "").strip(),
        "weather": request.args.get("weather", "").title().strip(),
        "activity": request.args.get("activity", "").strip(),
        "vibe": request.args.get("vibe", "").strip(),
    }
    active_filters = {key: value for key, value in selected_filters.items() if value}

    conn = get_db_connection()
    
    # We pull ALL 12 months for every city
    rows = conn.execute("""
        SELECT
            D.DestinationID AS destination_id,
            D.CityName AS destination,
            D.Country AS country,
            D.Description AS description,
            CP.BudgetLevel AS cost,
            WM.Month AS month,
            WM.AvgTempC AS temp,
            A.ActivityName AS activity_name,
            V.VibeName AS vibe
        FROM Destinations D
        LEFT JOIN Cost_Profiles CP ON CP.DestinationID = D.DestinationID
        LEFT JOIN Weather_Monthly WM ON WM.DestinationID = D.DestinationID
        LEFT JOIN Destinations_Activities DA ON DA.DestinationID = D.DestinationID
        LEFT JOIN Activities A ON A.ActivityID = DA.ActivityID
        LEFT JOIN Destination_Vibes DV ON DV.DestinationID = D.DestinationID
        LEFT JOIN Travel_Vibes V ON V.VibeID = DV.VibeID
    """).fetchall()
    conn.close()

    destinations = {}
    for row in rows:
        dest_id = row['destination_id']
        if dest_id not in destinations:
            destinations[dest_id] = {
                "destination": row['destination'],
                "country": row['country'],
                "description": row['description'],
                "cost": row['cost'],
                "activities": set(),
                "vibes": set(),
                "monthly_weather": {} 
            }

        if row['activity_name']: destinations[dest_id]["activities"].add(row['activity_name'])
        if row['vibe']: destinations[dest_id]["vibes"].add(row['vibe'])
        if row['month']: destinations[dest_id]["monthly_weather"][row['month']] = row['temp']

    results = []
    month_names = {1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May", 6:"Jun", 7:"Jul", 8:"Aug", 9:"Sep", 10:"Oct", 11:"Nov", 12:"Dec"}

    for dest_id, dest in destinations.items():
        weather = dest["monthly_weather"]
        
        def get_avg(months):
            temps = [weather[m] for m in months if m in weather]
            return round(sum(temps)/len(temps), 1) if temps else "N/A"

        dest["seasons"] = {
            "Winter": get_avg([12, 1, 2]),
            "Spring": get_avg([3, 4, 5]),
            "Summer": get_avg([6, 7, 8]),
            "Autumn": get_avg([9, 10, 11])
        }

        best_month_num = None
        user_weather_pref = selected_filters["weather"]
        
        if weather: 
            if user_weather_pref == "Warm":
                best_month_num = max(weather, key=weather.get) 
            elif user_weather_pref == "Cold":
                best_month_num = min(weather, key=weather.get)
            else: # Mild or unspecified defaults to ~20C
                best_month_num = min(weather, key=lambda m: abs(weather[m] - 20))

        if best_month_num:
            dest["best_month"] = month_names[best_month_num]
            dest["best_month_temp"] = weather[best_month_num]
        else:
            dest["best_month"] = "Unknown"
            dest["best_month_temp"] = "N/A"

        match_count = 0
        if normalize(selected_filters["cost"]) == normalize(dest["cost"]): match_count += 1
        if selected_filters["activity"] in dest["activities"]: match_count += 1
        if selected_filters["vibe"] in dest["vibes"]: match_count += 1
        
        if user_weather_pref: 
            if user_weather_pref == "Warm" and dest["best_month_temp"] != "N/A" and dest["best_month_temp"] >= 25:
                match_count += 1
            elif user_weather_pref == "Cold" and dest["best_month_temp"] != "N/A" and dest["best_month_temp"] <= 5:
                match_count += 1
            elif user_weather_pref == "Mild" and dest["best_month_temp"] != "N/A" and 10 < dest["best_month_temp"] < 25:
                match_count += 1

        dest["match_count"] = match_count
        dest["selected_filter_count"] = len(active_filters)
        
        dest["activities"] = list(dest["activities"])
        dest["vibes"] = list(dest["vibes"])
        del dest["monthly_weather"]
        
        results.append(dest)

    results.sort(key=lambda x: (-x["match_count"], x["destination"]))
    return jsonify(results)

# Master end point, sends all the data available.
@app.route('/api/database', methods=['GET'])
def get_full_database():
    conn = get_db_connection()
    
    # Grab everything from our core tables
    destinations = [dict(row) for row in conn.execute("SELECT * FROM Destinations").fetchall()]
    activities = [dict(row) for row in conn.execute("SELECT * FROM Activities").fetchall()]
    vibes = [dict(row) for row in conn.execute("SELECT * FROM Travel_Vibes").fetchall()]
    cost_profiles = [dict(row) for row in conn.execute("SELECT * FROM Cost_Profiles").fetchall()]
    
    # Limit weather to just 10 rows so we don't overwhelm the browser with 120 rows instantly
    weather = [dict(row) for row in conn.execute("SELECT * FROM Weather_Monthly LIMIT 12").fetchall()]
    
    conn.close()

    return jsonify({
        "Destinations": destinations,
        "Activities": activities,
        "Vibes": vibes,
        "Cost_Profiles": cost_profiles,
        "Weather_Monthly (Sample)": weather
    })

if __name__ == '__main__':
    app.run(port=5001, debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
127.0.0.1 - - [17/May/2026 19:56:36] "GET /api/filter-options HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 19:56:43] "GET /api/search?cost=Cheap&weather=Warm&activity=Beach+Vacation HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 19:56:46] "GET /api/search?cost=Cheap&weather=Warm&activity=Beach+Vacation HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 19:56:46] "GET /api/search?cost=Cheap&weather=Warm&activity=Beach+Vacation HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 19:56:49] "GET /api/search?cost=Cheap&activity=Beach+Vacation HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 19:56:51] "GET /api/search?cost=Cheap&weather=Mild&activity=Beach+Vacation HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 19:56:58] "GET /api/search?cost=Cheap&weather=Mild&activity=Beach+Vacation HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 19:57:00] "GET /api/search?weather=Mild&activity=Beach+Vacation HTTP/1.1" 200 -
127.0.0.1 - - [17/May/2026 19:57:04] "GET /api/filter-options H